In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

In [ ]:
def calculate_idx_position(account_capital_idr, risk_pct, entry_price, stop_loss_price):

    risk_amount_idr = account_capital_idr * (risk_pct / 100)

    if stop_loss_price <= 1:
      new_stop_loss_price = entry_price * (1-stop_loss_price)
      risk_per_share = entry_price - new_stop_loss_price
    else:
      risk_per_share = entry_price - stop_loss_price

    total_shares = risk_amount_idr / risk_per_share

    if total_shares < 100:
      total_lots = 0
    else :
      total_lots = int(total_shares // 100)

    position_value_idr = total_lots * 100 * entry_price

    print(f"--- IDX RISK MANAGEMENT REPORT ---")
    print(f"Account Capital : Rp {account_capital_idr:,.0f}")
    print(f"Stop Loss price : Rp {new_stop_loss_price}")
    print(f"Max Risk Allowed: Rp {risk_amount_idr:,.0f} ({risk_pct}%)")
    if total_lots == 0:
      print("Suggested Buy   : Lots are Low")
    else:
      print(f"Suggested Buy   : {total_lots} Lot(s) ({total_lots * 100} shares)")
    print(f"Capital Deployed: Rp {position_value_idr:,.0f}")

    return total_lots

In [ ]:
def calculate_phase2_ema_strategy(df, ema_short=25, ema_long=30, rsi_period=15):
    """
    Phase 2 Strategy: Fast Momentum Triggers
    1. EMA Crossover: Fast 12-day EMA crossing above Slow 26-day EMA.
    2. RSI Filter: Momentum health check to ensure RSI < 70 (not overbought).
    """
    # 1. Calculate Fast and Slow EMAs (To reduce lag and catch momentum early)
    df['EMA_Short'] = df['Close'].ewm(span=ema_short, adjust=False).mean()
    df['EMA_Long'] = df['Close'].ewm(span=ema_long, adjust=False).mean()

    # 2. Calculate RSI (Relative Strength Index) to screen out overbought conditions
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.ewm(alpha=1/rsi_period, min_periods=rsi_period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/rsi_period, min_periods=rsi_period, adjust=False).mean()

    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # 3. Generate Phase 2 Signals
    df['Signal'] = 0

    # Condition A: Fast EMA is greater than Slow EMA (bullish momentum)
    momentum_bullish = df['EMA_Short'] > df['EMA_Long']

    # Condition B: RSI is healthy (under 70)
    healthy_momentum = df['RSI'] < 70

    # Trigger Buy Signal (1) when both conditions align
    df.loc[momentum_bullish & healthy_momentum, 'Signal'] = 1

    return df

In [ ]:
# --- TEST EXECUTION (Phase 2 EMA Strategy) ---
if __name__ == "__main__":
    ticker = "BBCA.JK"
    print(f"Downloading historical data for {ticker}...")

    # Fetch 1 year of daily history for quick testing
    df = yf.download(ticker, period="1y", interval="1d")

    # Clean up yfinance multi-index columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)

    processed_df = calculate_phase2_ema_strategy(df)

    print(f"\n--- PHASE 2 EMA REPORT FOR {ticker} (LAST 5 DAYS) ---")
    print(processed_df[['Close', 'EMA_Short', 'EMA_Long', 'RSI', 'Signal']].tail())

/tmp/ipykernel_2366/421040123.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="1y", interval="1d")
[*********************100%***********************]  1 of 1 completed


--- PHASE 2 EMA REPORT FOR BBCA.JK (LAST 5 DAYS) ---
Price        Close    EMA_Short     EMA_Long        RSI  Signal
Date                                                           
2026-09-30  6075.0  6250.397464  6320.336391  36.533911       0
2026-10-01  6000.0  6211.874777  6296.607770  33.772152       0
2026-10-02  6100.0  6194.663273  6282.044231  40.257003       0
2026-10-05  6175.0  6191.638154  6274.115029  44.635609       0
2026-10-06  6150.0  6185.232284  6264.921323  43.491374       0
